# Цифровой детектив — обучение и инференс №121

**SegFormer MiT-B2 / decoder320, 640×640, epoch=32, адаптивный порог. Public AIC: 0.831234.**

Код обучения и инференса:
- Google Drive: https://drive.google.com/drive/folders/1Uwi7nqu8tlA3ZA7X-3U52LpLOUBnUPTz?usp=drive_link
- GitHub: https://github.com/zokost/aij26_final

Ноутбук загружает актуальную ветку main и запускает её модули. Весь код не требуется переносить в одну ячейку.
По умолчанию выполняется **инференс с готовыми весами**; полное обучение второго этапа включается отдельно.
Python3.10+, Git, CUDA. Для обучения нужны официальные данные, подготовленные NB-маски/CSV и инициализации stage1.

## 1. Пути и режим запуска
Измените пути к данным. Секреты в ноутбук не записываются; для приватного HF-репозитория можно заранее выполнить `hf auth login` в терминале.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile, io

CODE_URL = "https://github.com/zokost/aij26_final.git"
CODE_BRANCH = "main"
HF_REPO = "misiskovo/aij26-final-weights"
HF_REVISION = "e2b94832739c78c26818919ab83fb8177c8ad393"

WORKSPACE = (Path.cwd() / "work" / "reproduce").resolve()
DATA_ROOT = WORKSPACE / "data"
INFERENCE_ROOT = DATA_ROOT / "test_stage2"
INFERENCE_CSV = INFERENCE_ROOT / "test.csv"
PREPARED_ARCHIVE = Path.cwd() / "nc17_prepared_inputs.zip"
SOURCE_ARCHIVE = None  # Необязательно: Path("aij26_final_source_f3fcd99.zip") вместо GitHub.
LOCAL_WEIGHTS = Path.cwd() / "weights"  # Можно заранее положить сюда скачанные веса.

INSTALL_DEPENDENCIES = True
RUN_TRAINING = False
RESUME_TRAINING = False
RUN_INFERENCE = True
USE_RETRAINED_WEIGHTS = False  # False: точная посылка №121 из сданных весов.
GPU = "0"
TRAIN_GPUS = ["0"]  # Например, ["0", "1", "2"] для параллельного обучения учителей.

CODE_DIR = WORKSPACE / "code"
MANIFESTS_DIR = WORKSPACE / "manifests"
TRAIN_OUT = WORKSPACE / "training"
INFERENCE_OUT = WORKSPACE / "submission"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def run(arguments, cwd=CODE_DIR, env=None):
    arguments = [str(value) for value in arguments]
    print(" ".join(arguments), flush=True)
    subprocess.run(arguments, cwd=cwd, env=env, check=True)

def put_member(archive, member, destination):
    # Запись по явно проверенному относительному пути; существующие файлы не заменяются.
    if member.is_dir():
        return
    destination = Path(destination)
    if destination.exists():
        if sha256(destination) != hashlib.sha256(archive.read(member)).hexdigest():
            raise ValueError(f"Existing file differs: {destination}")
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    with archive.open(member) as src, destination.open("wb") as dst:
        shutil.copyfileobj(src, dst)

## 2. Исходники и зависимости
Загружается актуальная ветка `main`. Фактический хеш скачанного кода выводится в лог.
Опциональный архив исходников позволяет запустить заранее сохранённый снимок без GitHub и проверяется по SHA256.

In [ ]:
if SOURCE_ARCHIVE is not None:
    SOURCE_ARCHIVE = Path(SOURCE_ARCHIVE).resolve()
    if sha256(SOURCE_ARCHIVE) != "781fe5dcd35a6ca81240a66125379e48dd8ef09082c8e17e3b95100e9cfe8857":
        raise ValueError("Source archive SHA256 mismatch")
    with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
        for member in archive.infolist():
            relative = Path(member.filename)
            if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != "aij26_final":
                raise ValueError("Unsafe source archive path")
            put_member(archive, member, CODE_DIR.joinpath(*relative.parts[1:]))
else:
    fresh_checkout = not (CODE_DIR / ".git").is_dir()
    if fresh_checkout:
        run(["git", "clone", "--branch", CODE_BRANCH, "--single-branch", CODE_URL, CODE_DIR], cwd=WORKSPACE)
    if subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=CODE_DIR, text=True).strip() != CODE_URL:
        raise ValueError("Existing checkout has a different origin")
    if not fresh_checkout:
        run(["git", "diff", "--exit-code", "HEAD"])
    run(["git", "fetch", "origin", CODE_BRANCH])
    run(["git", "checkout", CODE_BRANCH])
    run(["git", "merge", "--ff-only", "origin/" + CODE_BRANCH])
    actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CODE_DIR, text=True).strip()
    print("Downloaded revision:", actual_revision)

if INSTALL_DEPENDENCIES:
    run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "huggingface_hub>=0.34,<3"])
print("Code:", CODE_DIR, "branch:", CODE_BRANCH)

## 3. Веса
Для инференса нужны только три файла `weights/121/` (около100МБ).
Для обучения дополнительно загружаются `FULL2_late_avg5.pt` и три больших stage1-учителя.
SHA256 каждого файла сверяется с `weights.json`. Повторное обучение не требуется для воспроизведения посылки.

In [ ]:
spec = json.loads((CODE_DIR / "weights.json").read_text())["files"]
needed = set()
if RUN_INFERENCE and not USE_RETRAINED_WEIGHTS:
    needed.update({"121/model.pt", "121/policy.pkl", "121/encoder_transform.pkl"})
if RUN_TRAINING:
    needed.update({"FULL2_late_avg5.pt", *[f"DIAG_teacher_fold{fold}of3_cnxl_832_e40.pt" for fold in range(3)]})
for item in spec:
    relative = item["file"]
    if relative not in needed:
        continue
    target = CODE_DIR / "weights" / relative
    if not target.exists():
        local = Path(LOCAL_WEIGHTS) / relative
        if local.is_file():
            source = local
        else:
            from huggingface_hub import hf_hub_download
            source = Path(hf_hub_download(repo_id=HF_REPO, filename="weights/" + relative, revision=HF_REVISION))
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
    if target.stat().st_size != item["bytes"] or sha256(target) != item["sha256"]:
        raise ValueError(f"Weight integrity check failed: {relative}")
    print("Verified:", relative)

## 4. Данные для обучения
Официальные RGB-изображения и stage2-маски предоставляются отдельно:
- `DATA_ROOT/stage2/train/{src,img,mask}/`;
- `DATA_ROOT/nanobanana/pico_nanobanana/{src,img}/`.

`nc17_prepared_inputs.zip` содержит фиксированные CSV и10243 разностные NB-маски, полученные из официальных пар; фотографий в нём нет.
Положите архив рядом с ноутбуком или укажите путь выше. Если `MANIFESTS_DIR` и NB-маски уже подготовлены, архив не требуется.
Мягкие цели учителей в архив не входят: они будут получены заново. Генерация исходных разностных масок в этот сценарий не входит.
Для одного инференса этот раздел не нужен.

In [ ]:
if RUN_TRAINING:
    if Path(PREPARED_ARCHIVE).is_file():
        if sha256(PREPARED_ARCHIVE) != "b7788a04f10155dafe8174e4615633d45b4b1faefcea64a8243ae48fe39625ce":
            raise ValueError("Prepared input archive SHA256 mismatch")
        manifest_spec = json.loads((CODE_DIR / "stage2/assets/manifests.json").read_text())
        with zipfile.ZipFile(PREPARED_ARCHIVE) as archive:
            manifest_bytes = archive.read(manifest_spec["file"])
            if hashlib.sha256(manifest_bytes).hexdigest() != manifest_spec["sha256"]:
                raise ValueError("Manifest archive SHA256 mismatch")
            with zipfile.ZipFile(io.BytesIO(manifest_bytes)) as lists:
                for member in lists.infolist():
                    relative = Path(member.filename)
                    if relative.is_absolute() or ".." in relative.parts or relative.parts[0] != "manifests":
                        raise ValueError("Unsafe manifest archive path")
                    put_member(lists, member, MANIFESTS_DIR.joinpath(*relative.parts[1:]))
            for member in archive.infolist():
                relative = Path(member.filename)
                if relative.parts[0] != "data":
                    continue
                if relative.is_absolute() or ".." in relative.parts:
                    raise ValueError("Unsafe data archive path")
                put_member(archive, member, DATA_ROOT.joinpath(*relative.parts[1:]))
    if not MANIFESTS_DIR.is_dir():
        raise FileNotFoundError("Provide nc17_prepared_inputs.zip or the prepared manifests directory")
    train_command = [sys.executable, "-m", "stage2.pipeline", "--root", DATA_ROOT,
                     "--manifests", MANIFESTS_DIR, "--weights", CODE_DIR / "weights",
                     "--gpus", *TRAIN_GPUS, "--out", TRAIN_OUT]
    run(train_command + ["--check-only"])
else:
    print("Training disabled: prepared training inputs are not required.")

## 5. Полное обучение второго этапа
Выполняется только при `RUN_TRAINING=True`:
1. Две тройки ConvNeXt-L/UPerNet-учителей, по12 эпох, из stage1-инициализаций.
2. Отложенные мягкие NB-маски и stage2-логиты.
3. MiT-B2 с новым декодером: до epoch32 по cosine-расписанию на40 эпох.
4. Обучение lower-only калибратора и NB-валидация.

`RESUME_TRAINING=True` продолжает прерванный пайплайн. Логи — `TRAIN_OUT/logs/`, готовый комплект — `TRAIN_OUT/final/`.
Обучение первого этапа отдельно описано в `stage1/README.md`; здесь используются его предоставленные инициализации.

In [ ]:
if RUN_TRAINING:
    if not TRAIN_GPUS:
        raise ValueError("Select at least one training GPU")
    run(train_command + (["--resume"] if RESUME_TRAINING else []))
else:
    print("Training skipped. Set RUN_TRAINING=True to train from stage-one initializations.")

## 6. Финальный инференс
Вход: каталог изображений и CSV с колонкой `img_path`, пути относительно `INFERENCE_ROOT`.
Выход: уникальная бинарная PNG-маска0/255 исходного размера для каждого изображения и `submission.zip`.
Используется исходный **FP32 batch4**, без TTA, SWA и повторного обучения.
`USE_RETRAINED_WEIGHTS=True` переключает инференс на новый комплект после обучения; он может отличаться от сданной посылки.

In [ ]:
if RUN_INFERENCE:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this inference implementation")
    if not Path(INFERENCE_CSV).is_file():
        raise FileNotFoundError(f"Provide input CSV: {INFERENCE_CSV}")
    calibration = TRAIN_OUT / "final" if USE_RETRAINED_WEIGHTS else CODE_DIR / "weights/121"
    checkpoint = calibration / "model.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    if INFERENCE_OUT.exists():
        raise FileExistsError(f"Choose a new INFERENCE_OUT to preserve existing predictions: {INFERENCE_OUT}")
    run([sys.executable, "-m", "stage2.infer", "--checkpoint", checkpoint,
         "--calibration", calibration, "--root", INFERENCE_ROOT,
         "--csv", INFERENCE_CSV, "--batch-size", "4", "--out", INFERENCE_OUT],
        env={**os.environ, "CUDA_VISIBLE_DEVICES": GPU})
else:
    print("Inference disabled.")

## 7. Проверка submission.zip
Проверяются порядок строк, уникальность масок, отсутствие лишних файлов, исходные размеры, режимL, значения0/255 и целостность ZIP.

In [ ]:
if RUN_INFERENCE:
    import numpy as np
    import pandas as pd
    from PIL import Image
    inputs = pd.read_csv(INFERENCE_CSV)
    submission_path = INFERENCE_OUT / "submission.zip"
    with zipfile.ZipFile(submission_path) as archive:
        if archive.testzip() is not None:
            raise ValueError("ZIP integrity check failed")
        submission = pd.read_csv(archive.open("submission.csv"))
        if list(submission.columns) != ["img_path", "prediction_path"]:
            raise ValueError("Unexpected submission columns")
        if submission.img_path.tolist() != inputs.img_path.tolist() or not submission.prediction_path.is_unique:
            raise ValueError("Submission coverage/order mismatch")
        expected = {"submission.csv", *submission.prediction_path.tolist()}
        if set(archive.namelist()) != expected or len(archive.namelist()) != len(expected):
            raise ValueError("Missing, duplicate or extra ZIP members")
        for row in submission.itertuples():
            with Image.open(Path(INFERENCE_ROOT) / row.img_path) as source_image:
                expected_size = source_image.size
            with Image.open(archive.open(row.prediction_path)) as mask:
                if mask.mode != "L" or mask.size != expected_size or not np.isin(np.asarray(mask), [0,255]).all():
                    raise ValueError(f"Invalid mask: {row.prediction_path}")
    print("Verified:", len(submission), "masks")
    print("Submission:", submission_path)
    print("SHA256:", sha256(submission_path))

## Материалы
- Детальный рецепт: `stage2/README.md`; проверка реализации: `python -m unittest discover -s tests -v`.
- EDA и сравнения: `stage2/eda_comparisons.ipynb`.
- Веса и хеши: `weights.json`. Оригинальные веса №121 используются без изменения.
- Код демо и скринкаст подаются отдельными материалами.